# 06 — SSP Poverty Predictions

Apply the best trained model for each poverty threshold to the full SSP forecast
panel (SSP1, SSP4, SSP5) and produce the prediction file consumed by the dashboard.

## Outputs

| File | Description |
|------|-------------|
| `data/final/poverty_predictions_ssp.csv` | Main dashboard feed: all countries × scenarios × years × thresholds |
| `outputs/top10_countries_by_scenario.csv` | Top-10 highest poverty countries per SSP × threshold × year |
| `outputs/prediction_summary_stats.csv` | Mean/median/std/p10/p90 by scenario × threshold × year |
| `outputs/prediction_trajectories_{threshold}.png` | Country-level trajectory charts |
| `outputs/prediction_global_heatmap.png` | Region-level poverty heatmap |

## Schema of `poverty_predictions_ssp.csv`

```
country_name        — IIASA country name
country_code        — ISO 3166-1 alpha-3
scenario            — SSP1 / SSP4 / SSP5
year                — 2025, 2030, …, 2100 (5-year steps)
poverty_threshold   — $3 / $4.20 / $8.30 / $10
predicted_poverty   — headcount ratio (%), clipped [0, 100]
approach            — A (year ≤ 2050) or B (year > 2050)
extrapolation_flag  — True if any input feature is source-extrapolated
```


## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from predict_ssp import (
    run_predictions,
    select_best_models,
    APPROACH_A_YEARS,
    APPROACH_B_YEARS,
    THRESHOLD_DISPLAY,
)
from config import DATA_FINAL_DIR, DATA_PROCESSED_DIR, OUTPUTS_DIR, SSP_SCENARIOS
from model_pipeline import ALL_THRESHOLDS, MODEL_NAMES
from utils import SCENARIO_COLORS

print("Approach A years:", APPROACH_A_YEARS)
print("Approach B years:", APPROACH_B_YEARS)

## 1. Confirm model training is complete

In [ ]:
from config import MODELS_DIR

model_files = list(MODELS_DIR.glob("*_approach_a.pkl"))
print(f"Model files found: {len(model_files)} (expected: {len(ALL_THRESHOLDS) * len(MODEL_NAMES)})")
for f in sorted(model_files):
    print(f"  {f.name}")

if len(model_files) == 0:
    print("\n⚠  No model files found. Run 03_model_training.ipynb first.")
else:
    print("\n✓ Model files present — ready to predict.")

## 2. Best-model selection

In [ ]:
best_models = select_best_models(OUTPUTS_DIR)
print("Best model per threshold (lowest test RMSE):")
for threshold, model in best_models.items():
    print(f"  {threshold:<8} → {model}")

# Show the comparison table for reference
csv_path = OUTPUTS_DIR / "model_comparison_by_threshold.csv"
if csv_path.exists():
    comp = pd.read_csv(csv_path)
    print("\nFull comparison (test RMSE):")
    print(
        comp[["model_name", "threshold", "rmse", "r2"]]
        .sort_values(["threshold", "rmse"])
        .to_string(index=False)
    )

## 3. Inspect the forecast panel

In [ ]:
panel = pd.read_csv(DATA_PROCESSED_DIR / "ssp_forecast_panel.csv")
print(f"Shape:     {panel.shape}")
print(f"Countries: {panel['country_name'].nunique()}")
print(f"Scenarios: {panel['scenario'].unique()}")
print(f"Years:     {panel['year'].min()} – {panel['year'].max()}")
print(f"Columns:   {panel.columns.tolist()}")

# Extrapolation flag summary
for col in ["employment_agriculture_extrap", "hdi_extrap", "coc_extrap"]:
    if col in panel.columns:
        n = panel[col].astype(bool).sum()
        yrs = sorted(panel[panel[col].astype(bool)]["year"].unique())
        print(f"  {col}: {n:,} rows  |  years: {yrs[:5]}…")

## 4. Run predictions

In [ ]:
# Generates all 4 thresholds × 3 scenarios × all years in one call.
predictions = run_predictions(
    forecast_panel_path = DATA_PROCESSED_DIR / "ssp_forecast_panel.csv",
    scaler_path         = DATA_FINAL_DIR     / "feature_scaler.pkl",
    feature_names_path  = DATA_FINAL_DIR     / "feature_names.json",
    models_dir          = MODELS_DIR,
    outputs_dir         = OUTPUTS_DIR,
    final_dir           = DATA_FINAL_DIR,
)
print(f"\nPredictions shape: {predictions.shape}")
predictions.head()

## 5. Verify output schema

In [ ]:
preds = pd.read_csv(DATA_FINAL_DIR / "poverty_predictions_ssp.csv")
print(f"Rows: {len(preds):,}")
print(f"Columns: {preds.columns.tolist()}")
print(f"\nDtype summary:")
print(preds.dtypes)
print(f"\nValue ranges:")
print(f"  predicted_poverty: [{preds['predicted_poverty'].min():.3f}, {preds['predicted_poverty'].max():.3f}]")
print(f"  extrapolation_flag: {preds['extrapolation_flag'].value_counts().to_dict()}")
print(f"  approach: {preds['approach'].value_counts().to_dict()}")
print(f"  scenarios: {preds['scenario'].unique()}")
print(f"  thresholds: {preds['poverty_threshold'].unique()}")
preds.head(10)

## 6. Prediction trajectory plots

In [ ]:
from IPython.display import Image, display
for threshold in ALL_THRESHOLDS:
    slug = threshold.replace("$", "").replace(".", "_")
    p = OUTPUTS_DIR / f"prediction_trajectories_{slug}.png"
    if p.exists():
        print(f"\n{THRESHOLD_DISPLAY[threshold]}")
        display(Image(str(p)))

## 7. Global heatmap

In [ ]:
p = OUTPUTS_DIR / "prediction_global_heatmap.png"
if p.exists():
    display(Image(str(p)))

## 8. Top-10 highest poverty countries

In [ ]:
top10 = pd.read_csv(OUTPUTS_DIR / "top10_countries_by_scenario.csv")
print("Top-10 by scenario/threshold/year:")
print(top10.head(30).to_string(index=False))

print("\nTop-10 highest poverty countries — $3/day, SSP4, year 2050:")
sub = top10[
    (top10["scenario"] == "SSP4") &
    (top10["poverty_threshold"] == "$3") &
    (top10["year"] == 2050)
]
print(sub[["rank","country_name","predicted_poverty","extrapolation_flag"]].to_string(index=False))

## 9. Summary statistics

In [ ]:
stats = pd.read_csv(OUTPUTS_DIR / "prediction_summary_stats.csv")

print("Global mean poverty — $3/day threshold:")
sub = stats[stats["poverty_threshold"] == "$3"][
    ["scenario", "year", "approach", "mean", "median", "p10", "p90", "n"]
].query("year in [2030, 2050, 2075, 2100]")
print(sub.to_string(index=False))

## 10. Scenario divergence analysis

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)

for ax, threshold in zip(axes, ALL_THRESHOLDS):
    sub = stats[stats["poverty_threshold"] == threshold]
    for ssp in SSP_SCENARIOS:
        g = sub[sub["scenario"] == ssp].sort_values("year")
        c = SCENARIO_COLORS.get(ssp, "grey")
        ax.plot(g["year"], g["mean"], color=c, linewidth=2, label=ssp)
        ax.fill_between(g["year"], g["p10"], g["p90"],
                        color=c, alpha=0.12)
    ax.axvline(2050, color="black", linestyle=":", linewidth=1)
    ax.axvspan(2051, 2100, alpha=0.05, color="red")
    ax.set_title(f"{THRESHOLD_DISPLAY[threshold]}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Year"); ax.set_ylabel("Poverty headcount (%)")
    ax.legend(fontsize=8); ax.grid(True, linestyle="--", alpha=0.35)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.suptitle(
    "Global mean poverty prediction (P10-P90 band) across countries\n"
    "Solid = Approach A | Faded = Approach B (extrapolated)",
    fontsize=10, y=1.02,
)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "scenario_divergence_all_thresholds.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 11. Approach A vs B comparison (prediction change after 2050)

In [ ]:
# For $3 / SSP1: show how much predictions change in the Approach B window
sub = preds[
    (preds["poverty_threshold"] == "$3") &
    (preds["scenario"] == "SSP1")
]

# Anchor at 2050
anchor = sub[sub["year"] == 2050][["country_name", "predicted_poverty"]]    .rename(columns={"predicted_poverty": "pred_2050"})
beyond = sub[sub["year"] > 2050].merge(anchor, on="country_name", how="left")
beyond["change_from_2050"] = beyond["predicted_poverty"] - beyond["pred_2050"]

drift_by_year = beyond.groupby("year")["change_from_2050"].agg(["mean","std"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(drift_by_year.index, drift_by_year["mean"],
       color="#1f77b4", alpha=0.75, label="Mean change from 2050 anchor")
ax.errorbar(drift_by_year.index, drift_by_year["mean"],
            yerr=drift_by_year["std"], fmt="none", color="black", capsize=3)
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Year"); ax.set_ylabel("Mean Δ poverty vs 2050 (pp)")
ax.set_title("$3/day — SSP1: Mean prediction drift in Approach B window")
ax.legend(); ax.grid(True, linestyle="--", alpha=0.35, axis="y")
plt.tight_layout(); plt.show()

## 12. Country-level sanity checks

In [ ]:
# Spot-check 6 countries spanning income levels
check_countries = ["Germany", "United States", "Brazil", "India", "Nigeria", "Ethiopia"]
available = preds["country_name"].unique()
check_countries = [c for c in check_countries if c in available]

print("Spot-check: predicted poverty $3/day, SSP1, selected years")
print(f"{'Country':<25}  {'2025':>7}  {'2030':>7}  {'2050':>7}  {'2075':>7}  {'2100':>7}")
print("-" * 65)
for country in check_countries:
    row_vals = {}
    for yr in [2025, 2030, 2050, 2075, 2100]:
        sub = preds[
            (preds["country_name"] == country) &
            (preds["poverty_threshold"] == "$3") &
            (preds["scenario"] == "SSP1") &
            (preds["year"] == yr)
        ]["predicted_poverty"]
        row_vals[yr] = f"{sub.values[0]:.1f}%" if len(sub) > 0 else " n/a "
    print(f"  {country:<23}  {row_vals[2025]:>7}  {row_vals[2030]:>7}  "
          f"{row_vals[2050]:>7}  {row_vals[2075]:>7}  {row_vals[2100]:>7}")

## 13. Final file listing

In [ ]:
main_out = DATA_FINAL_DIR / "poverty_predictions_ssp.csv"
aux_outs = [
    OUTPUTS_DIR / "top10_countries_by_scenario.csv",
    OUTPUTS_DIR / "prediction_summary_stats.csv",
    OUTPUTS_DIR / "prediction_global_heatmap.png",
    OUTPUTS_DIR / "scenario_divergence_all_thresholds.png",
] + [OUTPUTS_DIR / f"prediction_trajectories_{t.replace('$','').replace('.','_')}.png"
     for t in ALL_THRESHOLDS]

print("=== Final prediction outputs ===")
for p in [main_out] + aux_outs:
    if p.exists():
        print(f"  ✓  {p.name:<50}  {p.stat().st_size:>9,} bytes")
    else:
        print(f"  ✗  {p.name} — MISSING")

## 14. Summary

### `poverty_predictions_ssp.csv` — dashboard schema

The dashboard will read this file and filter on:
- `poverty_threshold` → dropdown selector (`$3` / `$4.20` / `$8.30` / `$10`)
- `scenario` → SSP1 / SSP4 / SSP5
- `year` → slider or dropdown
- `approach` → optional toggle to hide Approach B extrapolations
- `extrapolation_flag` → optional opacity overlay to signal uncertainty

Suggested aggregations for the dashboard:
- **Choropleth map**: `predicted_poverty` by `country_code` at a selected year
- **Line chart**: one line per scenario for a selected country
- **Bar chart**: top-N countries at a selected year
- **Uncertainty toggle**: hide `approach == "B"` for a conservative view

**Next**: `07_dashboard.ipynb` — Plotly Dash / Streamlit interactive visualisation.
